# Engineering Calculations — 100-House Steam Power Plant Digital Twin

This notebook is the calculation basis for the digital twin. It re-derives
every number used in the simulation (`/sim/rankine_cycle.py`) and the boiler
sizing referenced in `design-basis.md`, using the IAPWS-IF97 steam tables
(industry-standard formulation) via the `iapws` Python library.

**Purpose:** allow anyone reviewing this project to see the engineering basis
directly, not just trust a black-box simulation output.

**Sections:**
1. Plant load sizing
2. Rankine cycle state-point calculations
3. Mass flow & power solve
4. Cycle efficiency check against industry reference data
5. Fire-tube boiler sizing (tube count, heat transfer area)


## 1. Plant Load Sizing

Average household electrical demand (with diversity factor across lighting,
fans, appliances) is taken as 0.5–1 kW continuous. Applying a diversity
factor of ~0.6–0.7 to peak demand across 100 houses gives a target
**continuous electrical output of 75 kW**.

In [1]:
# Plant sizing inputs
NUM_HOUSES = 100
AVG_HOUSE_LOAD_KW = 1.0          # continuous average demand per house (pre-diversity)
DIVERSITY_FACTOR = 0.75          # accounts for non-coincident peak demand across houses

target_electrical_kw = NUM_HOUSES * AVG_HOUSE_LOAD_KW * DIVERSITY_FACTOR
print(f"Target electrical output: {target_electrical_kw:.1f} kW")


Target electrical output: 75.0 kW


## 2. Rankine Cycle State-Point Calculations

Cycle: **Boiler → Turbine → Condenser → Feedwater Pump → (back to Boiler)**

Design point:
- Boiler pressure: 15 bar(a)
- Turbine inlet temperature: 250°C (slight superheat)
- Condenser pressure: 0.2 bar(a)
- Turbine isentropic efficiency: 65%
- Generator efficiency: 92%
- Feed pump isentropic efficiency: 75%

State points use IAPWS-IF97 steam tables.

In [2]:
from iapws import IAPWS97

P_BOILER = 15.0                  # bar(a)
T_BOILER_C = 250.0               # deg C
P_COND = 0.2                     # bar(a)

ETA_TURBINE = 0.65
ETA_GEN = 0.92
ETA_PUMP = 0.75

T_BOILER_K = T_BOILER_C + 273.15

# --- State 1: Turbine inlet (boiler outlet) ---
s1 = IAPWS97(P=P_BOILER / 10, T=T_BOILER_K)   # iapws uses MPa for P
h1, s1_entropy = s1.h, s1.s
print(f"State 1 (turbine inlet): h1 = {h1:.1f} kJ/kg, s1 = {s1_entropy:.4f} kJ/kg-K")


State 1 (turbine inlet): h1 = 2924.0 kJ/kg, s1 = 6.7111 kJ/kg-K


In [3]:
# --- State 2s: Isentropic expansion to condenser pressure ---
s2s = IAPWS97(P=P_COND / 10, s=s1_entropy)
h2s = s2s.h

# --- State 2: Actual turbine outlet (accounting for isentropic efficiency) ---
h2 = h1 - ETA_TURBINE * (h1 - h2s)
s2 = IAPWS97(P=P_COND / 10, h=h2)
print(f"State 2s (isentropic):   h2s = {h2s:.1f} kJ/kg")
print(f"State 2 (actual outlet): h2  = {h2:.1f} kJ/kg, quality x = {s2.x:.3f}")


State 2s (isentropic):   h2s = 2210.4 kJ/kg
State 2 (actual outlet): h2  = 2460.1 kJ/kg, quality x = 0.937


In [4]:
# --- State 3: Condenser outlet (saturated liquid at P_COND) ---
s3 = IAPWS97(P=P_COND / 10, x=0)
h3 = s3.h
v3 = s3.v   # specific volume, m3/kg - needed for pump work
print(f"State 3 (condenser outlet): h3 = {h3:.1f} kJ/kg, T = {s3.T - 273.15:.1f} C")


State 3 (condenser outlet): h3 = 251.4 kJ/kg, T = 60.1 C


In [5]:
# --- State 4: Feedwater pump outlet (back to boiler pressure) ---
wp_ideal = v3 * (P_BOILER - P_COND) * 100   # kJ/kg  (bar * m3/kg * 100 = kJ/kg)
wp_actual = wp_ideal / ETA_PUMP
h4 = h3 + wp_actual
print(f"Pump work (ideal):  {wp_ideal:.2f} kJ/kg")
print(f"Pump work (actual): {wp_actual:.2f} kJ/kg")
print(f"State 4 (pump outlet): h4 = {h4:.1f} kJ/kg")


Pump work (ideal):  1.51 kJ/kg
Pump work (actual): 2.01 kJ/kg
State 4 (pump outlet): h4 = 253.4 kJ/kg


## 3. Mass Flow & Power Solve

Specific work and heat terms, then solve mass flow rate for the target
electrical output of 75 kW.

In [6]:
w_turbine = h1 - h2          # kJ/kg - turbine specific work
w_pump = wp_actual           # kJ/kg - pump specific work consumed
w_net = w_turbine - w_pump   # kJ/kg - net cycle work
q_boiler = h1 - h4           # kJ/kg - heat added in boiler
q_condenser = h2 - h3        # kJ/kg - heat rejected in condenser

print(f"Turbine specific work: {w_turbine:.1f} kJ/kg")
print(f"Net specific work:     {w_net:.1f} kJ/kg")
print(f"Boiler specific heat:  {q_boiler:.1f} kJ/kg")
print(f"Condenser specific heat: {q_condenser:.1f} kJ/kg")


Turbine specific work: 463.8 kJ/kg
Net specific work:     461.8 kJ/kg
Boiler specific heat:  2670.6 kJ/kg
Condenser specific heat: 2208.7 kJ/kg


In [7]:
# Solve mass flow rate for target electrical output:
# elec_kW = mdot * w_turbine * eta_gen - mdot * w_pump
TARGET_ELEC_KW = target_electrical_kw

mdot = TARGET_ELEC_KW / (w_turbine * ETA_GEN - w_pump)   # kg/s

turbine_shaft_kw = mdot * w_turbine
pump_power_kw = mdot * w_pump
elec_kw = turbine_shaft_kw * ETA_GEN - pump_power_kw
boiler_duty_kw = mdot * q_boiler
condenser_duty_kw = mdot * q_condenser

print(f"Mass flow rate: {mdot:.3f} kg/s ({mdot*3600:.0f} kg/hr)")
print(f"Turbine shaft output: {turbine_shaft_kw:.1f} kW")
print(f"Pump power consumption: {pump_power_kw:.2f} kW")
print(f"Net electrical output: {elec_kw:.1f} kW")
print(f"Boiler thermal duty: {boiler_duty_kw:.1f} kW ({boiler_duty_kw/1000:.3f} MW)")
print(f"Condenser heat rejection: {condenser_duty_kw:.1f} kW")


Mass flow rate: 0.177 kg/s (636 kg/hr)
Turbine shaft output: 81.9 kW
Pump power consumption: 0.35 kW
Net electrical output: 75.0 kW
Boiler thermal duty: 471.6 kW (0.472 MW)
Condenser heat rejection: 390.0 kW


## 4. Cycle Efficiency Check

Overall cycle efficiency and a sanity check against industry reference
ranges for small steam turbines (see `design-basis.md`, Section 5).

In [8]:
cycle_efficiency = elec_kw / boiler_duty_kw
print(f"Overall cycle efficiency: {cycle_efficiency*100:.2f}%")

print()
print("Reference context:")
print("- Large industrial steam turbines (2-250 MW, e.g. Siemens Energy): 30-45%+ overall efficiency")
print("- Small single-stage turbines can have isentropic efficiencies as low as 50%")
print("  (our 65% assumption is on the optimistic side for this scale)")
print("- Result is consistent with known efficiency penalties of scaling steam")
print("  turbines down to the tens-of-kW range, with no reheat/feedwater heating")


Overall cycle efficiency: 15.90%

Reference context:
- Large industrial steam turbines (2-250 MW, e.g. Siemens Energy): 30-45%+ overall efficiency
- Small single-stage turbines can have isentropic efficiencies as low as 50%
  (our 65% assumption is on the optimistic side for this scale)
- Result is consistent with known efficiency penalties of scaling steam
  turbines down to the tens-of-kW range, with no reheat/feedwater heating


## 5. Fire-Tube Boiler Sizing

Boiler type selected: **fire-tube (shell) boiler** — standard choice for
small industrial steam generation below a few tons/hour, matching our
~637 kg/hr requirement (water-tube boilers are reserved for larger
utility-scale, higher-pressure applications).

Estimate required heat transfer surface area and an approximate fire-tube
count using a typical small fire-tube boiler heat flux assumption.

In [9]:
# Typical fire-tube boiler heat flux (heat transfer rate per unit tube surface area)
# for small shell boilers - approximate design value from industry practice
TYPICAL_HEAT_FLUX_KW_PER_M2 = 25.0   # kW/m2, conservative for small fire-tube boilers

required_area_m2 = boiler_duty_kw / TYPICAL_HEAT_FLUX_KW_PER_M2
print(f"Required heat transfer area: {required_area_m2:.1f} m^2")


Required heat transfer area: 18.9 m^2


In [10]:
import math

# Assume standard small fire-tube outer diameter and tube length (typical packaged boiler range)
TUBE_OD_M = 0.05      # 50 mm OD - typical small fire-tube diameter
TUBE_LENGTH_M = 2.2   # matches boiler shell length assumption

area_per_tube = math.pi * TUBE_OD_M * TUBE_LENGTH_M
num_tubes_estimate = required_area_m2 / area_per_tube

print(f"Heat transfer area per tube: {area_per_tube:.3f} m^2")
print(f"Estimated number of fire-tubes required: {num_tubes_estimate:.0f}")
print()
print("Note: this is a first-order estimate for visual/modeling purposes")
print("(deciding fire-tube bundle density in the 3D model), not a certified")
print("boiler design calculation. Real boiler design would also account for")
print("gas-side velocity, fouling factors, and manufacturer-specific data.")


Heat transfer area per tube: 0.346 m^2
Estimated number of fire-tubes required: 55

Note: this is a first-order estimate for visual/modeling purposes
(deciding fire-tube bundle density in the 3D model), not a certified
boiler design calculation. Real boiler design would also account for
gas-side velocity, fouling factors, and manufacturer-specific data.


## Summary Table

| Parameter | Value |
|---|---|
| Target electrical output | 75.0 kW |
| Steam mass flow rate | ~0.177 kg/s |
| Turbine shaft output | ~81.9 kW |
| Boiler thermal duty | ~471.6 kW |
| Overall cycle efficiency | ~15.9% |
| Estimated fire-tube count | see calculation above |

This notebook is the authoritative calculation source for all numbers used
elsewhere in the project (`/sim/rankine_cycle.py`, `design-basis.md`, and
the boiler model in `/3d`).